# 03 – Reproducible Unsupervised Learning (v2)

This notebook runs `src/unsupervised_pipeline.py`, a single, reproducible version of the unsupervised stage.

**What changed compared with `02_Un_supervised_learning.ipynb`**

1. **One preprocessing object** (per-block StandardScaler → PCA → equal block weighting → global PCA 50D) is fitted on the training split only and applied identically to validation and test. In notebook 02 the validation and test features were produced by two different code paths, so their scores were not comparable.
2. **Feature consistency check.** `data/hsv_cache/test_features.npy` is not L2-normalised like train/valid (mean row norm 0.28 vs 1.00). The pipeline uses `test_features_original_pipeline.npy`, which matches train/valid.
3. **Label-free model selection.** K for K-Means is chosen by silhouette and GMM components by BIC on training data. Labels are used only for the final evaluation (ARI, NMI, purity, Hungarian accuracy).
4. **New methods:** Agglomerative (Ward), UMAP + K-Means, UMAP + HDBSCAN.
5. **Unsupervised disease detection:** one-class models trained on healthy leaves only flag diseased leaves as anomalies (ROC-AUC).
6. No hard-coded result numbers; every table is read from files the run creates.

In [ ]:
import sys, subprocess
from pathlib import Path
ROOT = Path.cwd().parent
OUT = ROOT / "results" / "unsupervised_learning" / "v2_reproducible"
# Full run takes about 3-5 min on a 2-core CPU. Add "--fast" for a quick check.
subprocess.run([sys.executable, str(ROOT / "src" / "unsupervised_pipeline.py")], check=True)

## Clustering results (labels used only after fitting)

In [ ]:
import pandas as pd, json
res = pd.read_csv(OUT / "clustering_results.csv")
display(res[res.Split == "test"].sort_values("NMI", ascending=False))
display(res[res.Split == "valid"].sort_values("NMI", ascending=False))
print(json.dumps(json.load(open(OUT / "summary.json")), indent=2))

## Label-free model selection

In [ ]:
display(pd.read_csv(OUT / "kmeans_selection.csv"))
display(pd.read_csv(OUT / "gmm_selection.csv"))

## Unsupervised disease detection (healthy vs diseased)

In [ ]:
display(pd.read_csv(OUT / "disease_detection_results.csv"))

## Figures

In [ ]:
from IPython.display import Image, display
for f in sorted(OUT.glob("*.png")):
    print(f.name); display(Image(filename=str(f)))